# Quality - Freshness Bronze

Valida que las tablas Bronze requeridas tengan datos recientes antes de ejecutar Silver/Gold incremental.

In [ ]:
from functools import reduce

from pyspark.sql import functions as F

try:
    dbutils.widgets.text('max_lag_days', '3')
    max_lag_days = int(dbutils.widgets.get('max_lag_days'))
except Exception:
    max_lag_days = 3

print(f'max_lag_days={max_lag_days}')

In [ ]:
checks = [
    {
        'source_name': 'nivel_ana',
        'table_name': 'weather.bronze.nivel_ana',
        'date_expr': 'to_date(to_timestamp(Data_Hora_Medicao))',
    },
    {
        'source_name': 'metar',
        'table_name': 'weather.bronze.metar',
        'date_expr': 'to_date(from_unixtime(cast(obsTime as bigint)))',
    },
    {
        'source_name': 'ana_rio_uruguai',
        'table_name': 'weather.bronze.ana_rio_uruguai',
        'date_expr': 'to_date(to_timestamp(Data_Hora_Medicao))',
    },
]

summary_dfs = []
for check in checks:
    summary_dfs.append(
        spark.sql(
            f'''
            SELECT
              '{check['source_name']}' AS source_name,
              '{check['table_name']}' AS table_name,
              MAX({check['date_expr']}) AS max_data_date,
              COUNT(*) AS rows_total
            FROM {check['table_name']}
            '''
        )
    )

summary = reduce(lambda left, right: left.unionByName(right), summary_dfs)
summary = summary.withColumn('allowed_min_date', F.date_sub(F.current_date(), max_lag_days))
summary = summary.withColumn('is_fresh', F.col('max_data_date').isNotNull() & (F.col('max_data_date') >= F.col('allowed_min_date')))

summary.show(truncate=False)

stale = summary.filter(~F.col('is_fresh'))
if stale.count() > 0:
    stale.show(truncate=False)
    raise ValueError('One or more Bronze tables are stale. Stop incremental Silver/Gold job.')